# Workflow

This notebook illustrates how to run simulations to create training data on the local machine.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal
lal.swig_redirect_standard_output_error(False);

In [ ]:
from cogwheel_machine import utils, generate_parameters, simulation, compression

## 1. Set up simulations directory and configuration file
Let us make a simulations directory with a copy of the example `config.py` in it.

After this you can edit the new config file as needed.

In [ ]:
datadir = '../data/'  # Edit as appropriate

In [ ]:
sim_dir = utils.setup_sim_dir(datadir)  # You may then edit the contents of the new config file

## 2. Generate simulation parameters

This will produce a dataframe `simulation_parameters.feather` with binary black hole parameters.

In [ ]:
generate_parameters.main(sim_dir)

Note: alternatively this could be run on a terminal like so:

    export OMP_NUM_THREADS=1
    python -m cogwheel_machine.generate_parameters {sim_dir}

## 3. Simulate signals and preprocess the data

This will produce the following files:
* `preprocessed_data.npz`: Auxiliary file with heterodyned data, heterodyned signals, and phenomenological reference waveform parameters.
* `folded_sampled_params.npy`: Contains the true (injected) parameter values, after folding.
* `unfolding_labels.npy`: Contains the true (injected) index of the unfolding transformation. 

In [ ]:
simulation.main(sim_dir, processes=None)

Note: alternatively this could be run on the terminal as

    export OMP_NUM_THREADS=1
    python -m cogwheel_machine.simulation {sim_dir}

## 4. Compress the data

This will create `compressed_data.npy`, which should be suitable for training the network.

In [ ]:
compression.create_mask(sim_dir)
compression.svd_compression(sim_dir)

Other compression algorithms (SVD, autoencoders) could be implemented as different functions in `compression.py`

Note: The compression is implemented as a step separate from preprocessing for the practical reason that the preprocessing is assumed to be trivially parallelizable (each preprocessed data can be computed in isolation from the other simulations).
On the other hand, compression algorithms might need to learn the distribution of the preprocessed data.